In [ ]:
Section 3: Hyperparameter Tuning and Model Comparison
This section will use GridSearchCV to tune model hyperparameters and compare performance across algorithms.
Random Forest was choosen for tuning since it performed best on dataset 1 and flexible for nonlinear relationships.

In [8]:
#Due to some issues, just made it into one cell block
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# DATASET 1: Gaming Academic Performance
df1 = pd.read_csv("Gaming_Academic_Performance.csv")
df1 = df1.drop(columns=["student_id"])
X1 = df1.drop(columns=["grades"])
y1 = df1["grades"]
numeric_features_1 = X1.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features_1 = X1.select_dtypes(include=["object"]).columns.tolist()
numeric_transformer_1 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])
categorical_transformer_1 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor_1 = ColumnTransformer(transformers=[
    ("num", numeric_transformer_1, numeric_features_1),
    ("cat", categorical_transformer_1, categorical_features_1)
])
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.2, random_state=42
)
# Original models for comparison
models_1 = {
    "Linear Regression": LinearRegression(),
    "KNN Regressor": KNeighborsRegressor(),
    "Random Forest Regressor": RandomForestRegressor(random_state=42)
}
results_1 = []
for name, model in models_1.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor_1),
        ("model", model)
    ])
    pipeline.fit(X1_train, y1_train)
    y1_pred = pipeline.predict(X1_test)
    results_1.append([
        name,
        mean_absolute_error(y1_test, y1_pred),
        np.sqrt(mean_squared_error(y1_test, y1_pred)),
        r2_score(y1_test, y1_pred)
    ])
results_1_df = pd.DataFrame(results_1, columns=["Model", "MAE", "RMSE", "R2"])

# Tune Random Forest for Dataset 1
rf_pipeline_1 = Pipeline(steps=[
    ("preprocessor", preprocessor_1),
    ("model", RandomForestRegressor(random_state=42))
])
param_grid_1 = {
    "model__n_estimators": [50, 100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5]
}
grid_search_1 = GridSearchCV(
    rf_pipeline_1,
    param_grid_1,
    cv=5,
    scoring="r2",
    n_jobs=-1
)
grid_search_1.fit(X1_train, y1_train)
best_tuned_1 = grid_search_1.best_estimator_
y1_tuned_pred = best_tuned_1.predict(X1_test)
dataset1_tuned_results = {
    "Model": "Tuned Random Forest",
    "MAE": mean_absolute_error(y1_test, y1_tuned_pred),
    "RMSE": np.sqrt(mean_squared_error(y1_test, y1_tuned_pred)),
    "R2": r2_score(y1_test, y1_tuned_pred)
}
print("=== Dataset 1 Best Parameters ===")
print(grid_search_1.best_params_)
print("=== Dataset 1 Best CV R2 ===")
print(grid_search_1.best_score_)

# Dataset 2: AI Student Life Dataset
df2 = pd.read_csv("AI_Impact_Student_Life_2026.csv")

# Create target variable
df2["GPA_Change"] = df2["GPA_Post_AI"] - df2["GPA_Baseline"]

#DropId
if "Student_ID" in df2.columns:
    df2 = df2.drop(columns=["Student_ID"])
# Remove leakage columns
X2 = df2.drop(columns=["GPA_Change", "GPA_Post_AI", "GPA_Baseline"])
y2 = df2["GPA_Change"]
numeric_features_2 = X2.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features_2 = X2.select_dtypes(include=["object"]).columns.tolist()
numeric_transformer_2 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])
categorical_transformer_2 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor_2 = ColumnTransformer(transformers=[
    ("num", numeric_transformer_2, numeric_features_2),
    ("cat", categorical_transformer_2, categorical_features_2)
])
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42
)
# Original models for comparison
models_2 = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=42),
    "Random Forest Regressor": RandomForestRegressor(random_state=42)
}
results_2 = []
for name, model in models_2.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor_2),
        ("model", model)
    ])
    pipeline.fit(X2_train, y2_train)
    y2_pred = pipeline.predict(X2_test)
    results_2.append([
        name,
        mean_absolute_error(y2_test, y2_pred),
        np.sqrt(mean_squared_error(y2_test, y2_pred)),
        r2_score(y2_test, y2_pred)
    ])
results_2_df = pd.DataFrame(results_2, columns=["Model", "MAE", "RMSE", "R2"])

# Tune Random Forest for Dataset 2
rf_pipeline_2 = Pipeline(steps=[
    ("preprocessor", preprocessor_2),
    ("model", RandomForestRegressor(random_state=42))
])
param_grid_2 = {
    "model__n_estimators": [50, 100, 200],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_split": [2, 5]
}
grid_search_2 = GridSearchCV(
    rf_pipeline_2,
    param_grid_2,
    cv=5,
    scoring="r2",
    n_jobs=-1
)
grid_search_2.fit(X2_train, y2_train)

best_tuned_2 = grid_search_2.best_estimator_
y2_tuned_pred = best_tuned_2.predict(X2_test)
dataset2_tuned_results = {
    "Model": "Tuned Random Forest",
    "MAE": mean_absolute_error(y2_test, y2_tuned_pred),
    "RMSE": np.sqrt(mean_squared_error(y2_test, y2_tuned_pred)),
    "R2": r2_score(y2_test, y2_tuned_pred)
}
print("\n=== Dataset 2 Best Parameters ===")
print(grid_search_2.best_params_)
print("=== Dataset 2 Best CV R2 ===")
print(grid_search_2.best_score_)

# COMPARISON TABLE
results_1_compare = results_1_df.copy()
results_1_compare["Dataset"] = "Gaming Academic Performance"

results_2_compare = results_2_df.copy()
results_2_compare["Dataset"] = "AI Student Life"

tuned_1_df = pd.DataFrame([dataset1_tuned_results])
tuned_1_df["Dataset"] = "Gaming Academic Performance"

tuned_2_df = pd.DataFrame([dataset2_tuned_results])
tuned_2_df["Dataset"] = "AI Student Life"

all_results = pd.concat(
    [results_1_compare, tuned_1_df, results_2_compare, tuned_2_df],
    ignore_index=True
)

print("\n=== All Model Results ===")
display(all_results.sort_values(by=["Dataset", "R2"], ascending=[True, False]))

=== Dataset 1 Best Parameters ===
{'model__max_depth': None, 'model__min_samples_split': 5, 'model__n_estimators': 200}
=== Dataset 1 Best CV R2 ===
0.9281511913228275

=== Dataset 2 Best Parameters ===
{'model__max_depth': 5, 'model__min_samples_split': 2, 'model__n_estimators': 200}
=== Dataset 2 Best CV R2 ===
-0.03602432080129523

=== All Model Results ===


,Model,MAE,RMSE,R2,Dataset
7,Tuned Random Forest,0.118086,0.138174,-0.019567,AI Student Life
4,Linear Regression,0.118876,0.138552,-0.025141,AI Student Life
6,Random Forest Regressor,0.119527,0.140338,-0.051745,AI Student Life
5,Decision Tree Regressor,0.162300,0.197682,-1.086877,AI Student Life
3,Tuned Random Forest,4.858230,6.273913,0.921472,Gaming Academic Performance
2,Random Forest Regressor,4.885081,6.307026,0.920641,Gaming Academic Performance
0,Linear Regression,5.503880,6.965510,0.903205,Gaming Academic Performance
1,KNN Regressor,6.903789,8.698109,0.849063,Gaming Academic Performance


In [ ]:
Hyperparameter Tuning and Model Comparison:
In this section, hyperparameter tuning was performed using GridSearchCV to 
optimize model performance and compare tuned results against the original 
baseline models. Random Forest was selected for tuning because it performed best 
on the Gaming Academic Performance dataset and is well-suited for 
capturing nonlinear relationships and feature interactions.

Dataset 1: Gaming Academic Performance
For the Gaming Academic Performance dataset, a Random Forest Regressor was tuned 
using the following hyperparameters:
- `n_estimators`: [50, 100, 200]
- `max_depth`: [None, 10, 20]
- `min_samples_split`: [2, 5]

The best parameter combination found by GridSearchCV was:
- `n_estimators = 200`
- `max_depth = None`
- `min_samples_split = 5`

The best 5-fold cross-validation score was:
- CV R² = 0.9282

After tuning, the model achieved the following test set performance:
- MAE = 4.8582
- RMSE = 6.2739
- R² = 0.9215

Compared to the original Random Forest model (**R² = 0.9206**), 
the tuned model showed a slight improvement. 
This indicates that the Gaming Academic Performance dataset has 
strong predictive structure and benefits modestly from hyperparameter optimization.

Dataset 2: AI Student Life
For the AI Student Life dataset, a Random Forest Regressor was also tuned 
using GridSearchCV. 
    
The following hyperparameters were tested:
- `n_estimators`: [50, 100, 200]
- `max_depth`: [None, 5, 10]
- `min_samples_split`: [2, 5]

The best parameter combination found was:
- `n_estimators = 200`
- `max_depth = 5`
- `min_samples_split = 2`

The best 5-fold cross-validation score was:
- CV R² = -0.0360

After tuning, the model achieved the following test set performance:
- MAE = 0.1181
- RMSE = 0.1382
- R² = -0.0196

Compared to the original Random Forest model R² = -0.0517, the tuned model improved 
slightly. However, the overall performance remained weak, which suggests that 
the available features may not strongly predict GPA change. This is likely because GPA 
change is a small and noisy target influenced by many external factors not captured in the dataset.

Overall Model Comparison
Across both datasets, hyperparameter tuning improved Random Forest performance, 
but the impact depended heavily on the strength of the dataset.

- The Gaming Academic Performance dataset showed strong predictive signal, and 
the tuned Random Forest achieved the best overall performance.
- The AI Student Life dataset showed limited predictive signal, and 
even after tuning, performance remained weak.

This comparison demonstrates that hyperparameter tuning can improve model performance, but 
it cannot fully overcome weak or noisy data. Model effectiveness depends not only on 
algorithm choice and parameter optimization, but also on the quality and predictiveness of the dataset itself.